# Step-1 : Install  the required libraries/packages

In [ ]:
#pip install pandas
#pip install openpyxl

# Step-2 : Import the required libraries/packages

In [1]:
import pandas as pd
import re
from urllib.parse import urlparse
import sqlite3


# Step 3: Load/Import data & Build connection

In [9]:
# Load Excel file
df = pd.read_excel("links.csv.xlsx", dtype=str)

# Replace NaN with empty string
df = df.fillna("")

# Connect to SQLite
conn = sqlite3.connect(":memory:")

# Load dataframe into SQLite
df.to_sql("links", conn, index=False, if_exists="replace")

cursor = conn.cursor()

print("Data loaded into SQLite")

Data loaded into SQLite


# Step - 4 : If the Links are in same format 


In [10]:
# Check datatypes

print('Below are the python data types', df.dtypes)
print('-----------------------------------------------------------------')

query = """
SELECT 
PRODUCT_ID,
URL_LINK,
substr(
        URL_LINK,
        instr(URL_LINK,'/drug/') + 6,
        instr(substr(URL_LINK, instr(URL_LINK,'/drug/') + 6), '/') - 1
     ) AS DRUG_NAME
FROM links;
"""

# Execute query
result_df = pd.read_sql_query(query, conn)

# Create output folder
output_folder = "output"
os.makedirs(output_folder, exist_ok=True)

# Output file path
file_path = os.path.join(output_folder, "extracted_drug_names.csv")

# Save result
result_df.to_csv(file_path, index=False)

print("Output saved in:", file_path)

Below are the python data types LINK_ID       str
PRODUCT_ID    str
GMID_CD       str
URL_LINK      str
dtype: object
-----------------------------------------------------------------
Output saved in: output\extracted_drug_names.csv


In [7]:
#!pip install wordfreq

#!pip install openpyxl

!pip install transformers torch pandas openpyxl

Defaulting to user installation because normal site-packages is not writeable
  Using cached transformers-5.3.0-py3-none-any.whl.metadata (32 kB)
  Using cached torch-2.10.0-cp313-cp313-win_amd64.whl.metadata (31 kB)
  Using cached tokenizers-0.22.2-cp39-abi3-win_amd64.whl.metadata (7.4 kB)
Using cached transformers-5.3.0-py3-none-any.whl (10.7 MB)
   ---------------------------------------- 0.0/618.0 kB ? eta -:--:--
   ---------------------------------------- 618.0/618.0 kB 4.8 MB/s  0:00:00
   ---------------------------------------- 0.0/3.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/3.7 MB ? eta -:--:--
   -------------- ------------------------- 1.3/3.7 MB 33.2 MB/s eta 0:00:01
   -------------- ------------------------- 1.3/3.7 MB 33.2 MB/s eta 0:00:01
   ------------------- -------------------- 1.8/3.7 MB 3.4 MB/s eta 0:00:01
   ---------------------- ----------------- 2.1/3.7 MB 3.1 MB/s eta 0:00:01
   ---------------------- ----------------- 2.1/3.7 MB 3.

ERROR: Could not install packages due to an OSError: [Errno 2] No such file or directory: 'C:\\Users\\U1109200\\AppData\\Local\\Packages\\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\\LocalCache\\local-packages\\Python313\\site-packages\\torch\\include\\ATen\\native\\transformers\\cuda\\mem_eff_attention\\iterators\\predicated_tile_access_iterator_residual_last.h'
HINT: This error might have occurred since this system does not have Windows Long Path support enabled. You can find information on how to enable this at https://pip.pypa.io/warnings/enable-long-paths



In [8]:
import pandas as pd
import re
from urllib.parse import urlparse
from transformers import pipeline
from wordfreq import zipf_frequency

# load dataset
df = pd.read_excel("drug_urls_varied_patterns.xlsx")

# biomedical NER model
ner = pipeline(
    "ner",
    model="d4data/biomedical-ner-all",
    aggregation_strategy="simple"
)

def extract_drug(url):

    path = urlparse(str(url)).path.lower()

    tokens = re.split(r'[-_/0-9.]', path)

    tokens = [t for t in tokens if len(t) > 3]

    # 1️⃣ Try NER detection
    for token in tokens:
        result = ner(token)
        for ent in result:
            if ent.get("entity_group") in ["CHEMICAL","DRUG"]:
                return token

    # 2️⃣ fallback: rare-word detection
    rare_words = [w for w in tokens if zipf_frequency(w, "en") < 3]

    if rare_words:
        return max(rare_words, key=len)

    return None


df["drug_name"] = df["url"].apply(extract_drug)

df.to_csv("drug_extracted_output.csv", index=False)

print(df)

ModuleNotFoundError: No module named 'transformers'